# Inference-Mirror GRPO Training (Colab)

Train **Qwen2.5-1.5B-Instruct** on the Bioresearch OpenEnv environment with GRPO, mirroring [`inference.py`](https://huggingface.co/spaces/anirudhchida/bioresearch/blob/main/inference.py) verbatim — same system prompts, same parsers, same per-turn prompt builders, same multi-turn lab rollout up to `TRAIN_LAB_MAX_STEPS` tool-calling turns per episode.

**What is different from the sibling [`train_grpo_colab.ipynb`](train_grpo_colab.ipynb)?**

- **All 14 tasks are trainable in one pass** via task-type-gated reward functions — each task gets its own reward curve on the same plot.
- **Lab tasks do a real multi-turn rollout** inside the reward function (Turn 1 is the TRL-generated completion and carries the gradient; Turns 2..N are unrolled under `torch.inference_mode`). This mirrors `inference.py._run_lab_episode` instead of the cheaper immediate-submit.
- **A shared [`training_core.py`](../training_core.py)** module holds all the heavy logic. This notebook is a thin driver; [`tests/test_training_core.py`](../tests/test_training_core.py) smoke-tests the same code paths without a GPU.

**Hardware.** Defaults are tuned for a free Colab **T4 16 GB** running ~60 minutes on 6 tasks. Flip `TASK_LIST = ALL_TASKS` on an A100 for the full 14-curve version.

## 1 · Install dependencies

In [ ]:
# Pin pydantic<2.12 — pydantic 2.12 imports `TypeForm` from typing_extensions,
# which Colab's preinstalled typing_extensions 4.12 doesn't have, producing the
# `ImportError: cannot import name 'TypeForm'` you hit when `import training_core`
# transitively pulls pydantic in. Upgrading typing_extensions on top is a
# belt-and-suspenders guard for any other dep that wants the newer API.
!pip install -q -U "typing_extensions>=4.15"
!pip install -q "pydantic<2.12"

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.26" "trl<0.11" peft accelerate bitsandbytes
!pip install -q httpx fastapi uvicorn pandas matplotlib datasets

## 2 · Clone the environment repo (Colab)

In [ ]:
import os, subprocess, sys

REPO_URL = os.environ.get(
    "BIORESEARCH_REPO_URL",
    "https://huggingface.co/spaces/anirudhchida/bioresearch",
)

if not os.path.isdir("bioresearch"):
    if "anirudhchida" in REPO_URL:
        raise RuntimeError(
            "Set BIORESEARCH_REPO_URL (or edit REPO_URL above) to your fork — "
            "the placeholder URL will 404 on clone."
        )
    subprocess.run(["git", "clone", REPO_URL, "bioresearch"], check=True)

sys.path.insert(0, "/content/bioresearch")
print("Setup complete")

## 3 · Boot the OpenEnv server

The reward loop talks HTTP to this subprocess. We poll `/health` until uvicorn is serving — current OpenEnv releases expose `/health`, `/schema`, `/docs`, and `/openapi.json`, not the legacy `/info`.

In [ ]:
import subprocess, time, httpx

server_proc = subprocess.Popen(
    ["uvicorn", "server.app:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd="/content/bioresearch",
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

for _ in range(40):
    try:
        if httpx.get("http://127.0.0.1:8000/health", timeout=1.0).status_code == 200:
            print("Server up")
            break
    except Exception:
        time.sleep(1.0)
else:
    raise RuntimeError("OpenEnv server failed to start")

## 4 · Configure the shared `training_core` module

`training_core` owns prompts, parsers, dataset builder, reward-function factory, and the multi-turn lab rollout. We register the server URL here; the model is registered in Cell 7 after Unsloth finishes loading it.

In [ ]:
import training_core
from training_core import (
    LEGACY_TASKS, LAB_TASKS, ALL_TASKS, DEFAULT_T4_TASKS,
    TRAIN_LAB_MAX_STEPS, EVAL_LAB_MAX_STEPS,
)

SERVER_URL = "http://127.0.0.1:8000"
training_core.configure_env(SERVER_URL)

probe = training_core.env_reset(task_type="dna_classification").observation
print(f"Probe OK  task_id={probe.task_id}  question[:60]={probe.question[:60]!r}")

print(f"\nTRAIN_LAB_MAX_STEPS = {TRAIN_LAB_MAX_STEPS}  (per-task-training multi-turn budget)")
print(f"EVAL_LAB_MAX_STEPS  = {EVAL_LAB_MAX_STEPS}  (held-out eval budget, closer to inference.py's 20)")

## 5 · Choose the task list

The default is six tasks — five single-step + one lab — which fits on a T4 in ~60 minutes at `max_steps=200`. Swap to `ALL_TASKS` on an A100 for the full 14-curve run. Any subset of `ALL_TASKS` works; each listed task gets its own reward function and its own curve in the log history.

In [ ]:
# Default (T4-friendly):
TASK_LIST = DEFAULT_T4_TASKS

# Full 14-task coverage (A100, ~3h):
# TASK_LIST = ALL_TASKS

# Custom subset example:
# TASK_LIST = ["dna_classification", "perturbation_direction_qa", "protein_hypothesis_lab"]

print("Training on:", TASK_LIST)
print("  legacy:", [t for t in TASK_LIST if t in LEGACY_TASKS])
print("  lab   :", [t for t in TASK_LIST if t in LAB_TASKS])

## 6 · Build the mixed-task dataset

Each row is `{prompt: [system, user], task_id, task_type}`. TRL forwards `task_id` and `task_type` through `**kwargs` to the reward functions — that is what lets each reward function pin the env to the *exact* brief its completion was graded against. Without this alignment the GRPO advantage collapses into noise.

In [ ]:
N_PER_TASK = 8
train_ds = training_core.build_mixed_dataset(TASK_LIST, n_per_task=N_PER_TASK, seed=42)
print(train_ds)
print("Sample row keys:", train_ds.column_names)

## 7 · Load Qwen2.5-1.5B with Unsloth and register it with `training_core`

Unsloth loads the model in 4-bit and wraps a LoRA adapter in one call. After loading we call `training_core.configure_model(...)` so the lab-task reward function can generate Turns 2..N through the same model.

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
tokenizer.padding_side = "left"

training_core.configure_model(model, tokenizer, max_new_tokens=256)
print("Model + tokenizer loaded and registered with training_core")

## 8 · Build one reward function per task

Each closure scores only rows whose `task_type` matches it — rows from other tasks return `0.0`. That is how we get N non-overlapping reward curves out of a single `GRPOTrainer`.

In [ ]:
reward_fns = [training_core.make_reward_fn(t) for t in TASK_LIST]
print("Reward functions wired up:")
for fn in reward_fns:
    print(" -", fn.__name__)

## 9 · `GRPOConfig` + `GRPOTrainer`

Hyperparameters tuned for T4 + Qwen2.5-1.5B. Key knobs:

- `num_generations=4` — group size for GRPO's relative advantage. Drop to 2 if you OOM.
- `learning_rate=5e-6` — raising this past 1e-5 tends to diverge on 1.5B within ~50 steps.
- `max_completion_length=512` — higher than `train_grpo_colab.ipynb`'s 256 because lab *turns* need more room than immediate-submits.
- `beta=0.04` — KL anchor. Raise to 0.08 if the policy starts emitting gibberish.

In [ ]:
import torch
from trl import GRPOConfig, GRPOTrainer

_BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

grpo_config = GRPOConfig(
    output_dir="grpo_bioresearch_mirror",
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=1024,
    max_completion_length=512,
    max_steps=200,
    logging_steps=1,
    save_strategy="steps",
    save_steps=50,
    bf16=_BF16_OK,
    fp16=not _BF16_OK,
    beta=0.04,
    max_grad_norm=1.0,
    seed=42,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=grpo_config,
    train_dataset=train_ds,
    reward_funcs=reward_fns,
)
print("Trainer ready")

## 10 · Train

In [ ]:
trainer.train()

## 11 · Plot per-task reward curves

Each reward function produces its own column in `trainer.state.log_history` (TRL names it `rewards/<fn_name>`). Per-task curves are only meaningful on the rows for that task, so for each task we mask out `0.0` values (the "gated-out" rows from other tasks) before smoothing.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

log_df = pd.DataFrame(trainer.state.log_history)
print("Logged columns:", [c for c in log_df.columns if "reward" in c])

fig, ax = plt.subplots(figsize=(10, 5))
for task in TASK_LIST:
    candidates = [c for c in log_df.columns
                  if task in c and "reward" in c and "std" not in c.lower()]
    if not candidates:
        print(f"  (no reward column found for {task})")
        continue
    col = candidates[0]
    subset = log_df[["step", col]].dropna()
    subset = subset[subset[col] > 0.0].reset_index(drop=True)
    if subset.empty:
        continue
    smooth = subset[col].rolling(window=10, min_periods=1).mean()
    ax.plot(subset["step"], smooth, linewidth=2.0, label=task)

ax.set_xlabel("GRPO step")
ax.set_ylabel("Reward (10-step EMA, gated rows only)")
ax.set_title("Per-task reward curves — Inference-Mirror GRPO")
ax.legend(loc="best", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("grpo_reward_curves_mirror.png", dpi=140)
plt.show()

## 12 · Held-out evaluation (`inference.py` parity)

Run each task on 5 fresh deterministic `task_id`s the trainer has *never* seen. `training_core.run_eval_episode` uses the exact same rollout shape as `inference.py` — single-turn for legacy tasks, multi-turn (up to `EVAL_LAB_MAX_STEPS`) for lab tasks. Results get written to `eval_summary_mirror.json`.

In [ ]:
import json
import numpy as np

N_EVAL = 5
FastLanguageModel.for_inference(model)

eval_summary = {}
for task in TASK_LIST:
    seen = []
    attempts = 0
    while len(seen) < N_EVAL and attempts < N_EVAL * 4:
        attempts += 1
        obs = training_core.env_reset(task_type=task).observation
        if obs.task_id and obs.task_id not in seen:
            seen.append(obs.task_id)

    rewards = []
    for tid in seen:
        r = training_core.run_eval_episode(tid, task)
        rewards.append(r)
    mean, std = float(np.mean(rewards)), float(np.std(rewards))
    eval_summary[task] = {"n": len(rewards), "mean": mean, "std": std, "rewards": rewards}
    print(f"  {task:30s}  n={len(rewards)}  mean={mean:.3f}  std={std:.3f}")

with open("eval_summary_mirror.json", "w") as f:
    json.dump(eval_summary, f, indent=2)
print("\nWrote eval_summary_mirror.json")

FastLanguageModel.for_training(model)

## 13 · Save the LoRA adapter + teardown

In [ ]:
model.save_pretrained("grpo_bioresearch_mirror_lora")
tokenizer.save_pretrained("grpo_bioresearch_mirror_lora")
print("LoRA saved to grpo_bioresearch_mirror_lora/")

server_proc.terminate()
try:
    server_proc.wait(timeout=5)
except Exception:
    server_proc.kill()
print("Server terminated")